In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle



In [2]:
model=load_model("model.h5")
with open("onehot_geo.pkl","rb") as file:
    onehot_geo=pickle.load(file)
with open("label_encoder_gender.pkl","rb") as file:
    label_encoder_gender=pickle.load(file)
with open("scaler.pkl","rb")as file:
    scaler=pickle.load(file)

In [3]:
input_data={
    'CreditScore': 700,
    'Geography': 'Germany',
    'Gender':'Male',
    'Age':25,
    'Tenure':3,
    'Balance': 600000,
    'NumOfProducts':2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimateSalary': 500000
}

In [4]:
geo_encoder=onehot_geo.transform([[input_data["Geography"]]])
geo_encoder
geo_encoder_df=pd.DataFrame(geo_encoder,columns=onehot_geo.get_feature_names_out(["Geography"]))
geo_encoder_df

d:\DL_project\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,0.0,1.0,0.0


In [5]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimateSalary
0,700,Germany,Male,25,3,600000,2,1,1,500000


In [6]:
# encode categorical variables
input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])

In [7]:
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimateSalary
0,700,Germany,1,25,3,600000,2,1,1,500000


In [8]:
input_df=pd.concat([input_df.drop("Geography",axis=1),geo_encoder_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimateSalary,Geography_France,Geography_Germany,Geography_Spain
0,700,1,25,3,600000,2,1,1,500000,0.0,1.0,0.0


In [12]:
input_df=input_df.rename(columns={'EastimatedSalary':'EstimatedSalary'})

In [13]:
# scaling the input data
input_scaled=scaler.transform(input_df)
input_scaled

array([[ 0.53765312,  0.87920816, -1.46950731, -0.71863972,  8.62170112,
         0.81495449,  0.57551132,  1.00483224,  7.6434953 , -1.14292038,
         1.92890808, -0.53381936]])

In [14]:
prediction=model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step


array([[0.71330255]], dtype=float32)

In [15]:
prediction_probability=prediction[0][0]
prediction_probability

np.float32(0.71330255)

In [17]:
if prediction_probability>0.5:
    print("The customer is likely to churn")
else:
    print("The customer is not churn") 

The customer is likely to churn
